# Эксперименты с эмбеддерами
Примечание: скоринг одного сэтапа занимает время

In [1]:
import requests
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
from torchvision.datasets import CIFAR100, EMNIST, STL10
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from transformers import (
    AutoTokenizer, AutoModel,
    AutoImageProcessor, ResNetModel,
    CLIPProcessor, CLIPModel,
    AutoModelForCausalLM, BitsAndBytesConfig,
    AutoImageProcessor, SwinModel
)
from urllib.request import urlopen
from huggingface_hub import hf_hub_download
from datasets import load_dataset

from sklearn.cluster import DBSCAN, KMeans
from sklearn import metrics

# Datasets

## CIFAR

In [2]:
# впервые download=True
cifar100 = CIFAR100('../data/cifar', download=True, train=False)
cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100]
cifar_img = [img[0] for img in cifar100]
cifar_labels = [obj[1] for obj in cifar100]

print(cifar100)
len(cifar100.classes)

Dataset CIFAR100
    Number of datapoints: 10000
    Root location: ../data/cifar
    Split: Test


100

## EMNIST Digits

In [3]:
emnist = EMNIST('../data/emnist', download=True, train=False, split='digits')
emnist_txt = [f'a picture of number {emnist.classes[img[1]]}' for img in emnist][:10000]
emnist_img = [img[0] for img in emnist][:10000]
emnist_img = [img.convert('RGB') for img in emnist_img]
emnist_labels = [obj[1] for obj in emnist][:10000]

100%|██████████| 562M/562M [00:02<00:00, 197MB/s]  


## MSCOCO

In [4]:
mscoco = load_dataset('clip-benchmark/wds_mscoco_captions2017', split="test")
mscoco_img = mscoco['jpg']
mscoco_img = [img.convert('RGB') for img in mscoco_img]
mscoco_txt = mscoco['txt']

mscoco

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['__key__', '__url__', 'jpg', 'txt'],
    num_rows: 5000
})

# Models

- CLIP Model (large, small)
- SWIN Transformer и E5 small
- get_resnet50 и E5 small - baseline

In [5]:
def get_clip_embeddings(model_name: str, img_inputs, txt_inputs, batch_size: int = 32):
    """Получение CLIP эмбеддингов
    
    Args:
        model_name (str): название модели: "openai/clip-vit-large-patch14", "openai/clip-vit-base-patch32", "../models/CLIP-GmP-ViT-L-14"
        img_inputs: список изображений
        txt_inputs: список текстов
        batch_size (int): размер батча для обработки
    """
    # Проверяем доступность CUDA
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Загружаем модель и процессор
    model = CLIPModel.from_pretrained(model_name).to(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    
    # Устанавливаем модель в режим оценки
    model.eval()

    # Предварительное выделение памяти для эмбеддингов
    with torch.no_grad():
        # Получаем размерность эмбеддингов на тестовом батче
        test_inputs = processor(
            text=txt_inputs[:1],
            images=img_inputs[:1],
            return_tensors="pt"
        ).to(device)
        
        test_outputs = model(**test_inputs)

        text_embed_dim = test_outputs.text_embeds.shape[1]
        image_embed_dim = test_outputs.image_embeds.shape[1]
        total_embed_dim = text_embed_dim + image_embed_dim
        embeds = np.zeros((len(txt_inputs), total_embed_dim))

        # Обработка батчами
        for i in range(0, len(txt_inputs), batch_size):
            batch_text = txt_inputs[i:i+batch_size]
            batch_images = img_inputs[i:i+batch_size]

            inputs = processor(
                text=batch_text,
                images=batch_images,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            outputs = model(**inputs)

            # Конкатенация и сохранение эмбеддингов
            batch_embeds = torch.cat(
                (outputs.text_embeds, outputs.image_embeds),
                dim=1
            ).cpu().numpy()

            embeds[i:i+batch_size] = batch_embeds
        
    # Очищаем кеш CUDA
    del model, processor
    torch.cuda.empty_cache()
    
    return embeds

In [6]:
def get_resnet50_e5small_embeddings(img_inputs, txt_inputs, batch_size=32):
    """Получение resnet50 и e5small эмбеддингов с батчевой обработкой"""
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
        last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
        return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    
    # Инициализация моделей
    resnet_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
    resnet_model = ResNetModel.from_pretrained("microsoft/resnet-50").to(device).eval()

    e5_tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
    e5_model = AutoModel.from_pretrained('intfloat/multilingual-e5-small').to(device).eval()

    # Предварительное выделение памяти
    num_samples = len(txt_inputs)
    final_embeds = np.zeros((num_samples, 384 + 2048))
    
    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            batch_end = min(i + batch_size, num_samples)
            
            # Подготовка батча текстов
            batch_texts = txt_inputs[i:batch_end]
            e5_inputs = e5_tokenizer(
                batch_texts, 
                max_length=512, 
                padding=True, 
                truncation=True, 
                return_tensors='pt'
            ).to(device)
            
            # Получение текстовых эмбеддингов
            e5_outputs = e5_model(**e5_inputs)
            e5_embeddings = average_pool(e5_outputs.last_hidden_state, e5_inputs['attention_mask'])
            e5_embeddings = F.normalize(e5_embeddings, p=2, dim=1)
            
            # Подготовка и обработка изображений
            resnet_embeddings = []
            for img in img_inputs[i:batch_end]:
                inputs = resnet_processor(img, return_tensors="pt").to(device)
                outputs = resnet_model(**inputs)
                
                # Явное преобразование размерности
                if outputs.pooler_output.dim() > 2:
                    pooler_output = outputs.pooler_output.squeeze()
                else:
                    pooler_output = outputs.pooler_output
                
                # Проверка размерности
                if pooler_output.dim() == 1:
                    pooler_output = pooler_output.unsqueeze(0)
                
                resnet_embeddings.append(pooler_output)
            
            resnet_embeddings = torch.cat(resnet_embeddings, dim=0)
            
            # Конкатенация результатов
            batch_result = torch.cat(
                [e5_embeddings, resnet_embeddings], 
                dim=1
            ).cpu().numpy()
            
            final_embeds[i:batch_end] = batch_result
    
    torch.cuda.empty_cache()
    return final_embeds

In [7]:
def get_swin_e5small_embeddings(img_inputs, txt_inputs, batch_size=32):
    """Получение swin_transformer и e5small эмбеддингов с батчевой обработкой"""
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
        last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
        return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    
    # Инициализация моделей
    swin_processor = AutoImageProcessor.from_pretrained("microsoft/swin-base-patch4-window7-224-in22k")
    swin_model = SwinModel.from_pretrained("microsoft/swin-base-patch4-window7-224-in22k").to(device).eval()

    e5_tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
    e5_model = AutoModel.from_pretrained('intfloat/multilingual-e5-small').to(device).eval()

    # Предварительное выделение памяти
    num_samples = len(txt_inputs)
    final_embeds = np.zeros((num_samples, 384 + 1024))
    
    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            batch_end = min(i + batch_size, num_samples)
            
            # Подготовка батча изображений
            batch_images = img_inputs[i:batch_end]
            swin_inputs = swin_processor(
                batch_images, 
                return_tensors="pt"
            ).to(device)
            
            # Подготовка батча текстов
            batch_texts = txt_inputs[i:batch_end]
            e5_inputs = e5_tokenizer(
                batch_texts, 
                max_length=512, 
                padding=True, 
                truncation=True, 
                return_tensors='pt'
            ).to(device)
            
            # Получение эмбеддингов
            e5_outputs = e5_model(**e5_inputs)
            e5_embeddings = average_pool(e5_outputs.last_hidden_state, e5_inputs['attention_mask'])
            e5_embeddings = F.normalize(e5_embeddings, p=2, dim=1)
            
            swin_outputs = swin_model(**swin_inputs).pooler_output
            
            # Конкатенация результатов
            batch_result = torch.cat(
                [e5_embeddings, swin_outputs], 
                dim=1
            ).cpu().numpy()
            
            final_embeds[i:batch_end] = batch_result
    
    torch.cuda.empty_cache()
    return final_embeds

## Кластеризация

In [8]:
def get_clustering_metrics(X, labels, target):
    return [
        # Внутренние меры
        metrics.silhouette_score(X, labels, metric='euclidean'),
        metrics.calinski_harabasz_score(X, labels),
        metrics.davies_bouldin_score(X, labels),
        
        # Внешние меры
        metrics.rand_score(target, labels) if target else 0,
        metrics.normalized_mutual_info_score(target, labels) if target else 0
    ]

def get_clustering_experiment(exp_name: str, img_inputs, txt_inputs,
                              preprocessor_name: str, data_name: str,
                              df_results: pd.DataFrame = None,
                              model_name: str = None, target = None):
    
    if data_name == "cifar":
        kmeans = KMeans(n_clusters=100)
    else:
        kmeans = KMeans(n_clusters=10)
    
    print('exp_name')
    print('preprocessor step')
    if preprocessor_name == "clip":
        embs = get_clip_embeddings(model_name, img_inputs, txt_inputs, batch_size=32)
    if preprocessor_name == "baseline":
        embs = get_resnet50_e5small_embeddings(img_inputs, txt_inputs)
    if preprocessor_name == "swin_e5":
        embs = get_swin_e5small_embeddings(img_inputs, txt_inputs)
    
    print('preprocessor kmeans step')
    kmeans.fit(embs)
    
    if df_results is None:
        df_results = pd.DataFrame(columns=['method-data', 
                                           'silhouette', 'calinski_harabasz', 'davies_bouldin_score',
                                           'rand_score', 'normalized_mutual_info_score'])
    
    print('calculate metrics step')
    df_results.loc[df_results.shape[0]] = [exp_name]+get_clustering_metrics(embs, kmeans.labels_, target)
    
    
    return df_results

### Для проверки всех сетапов нужно раскоментировать - возмо

In [9]:
exp_setup = {
    "clip_l-CIFAR": {
        "exp_name": "clip_l-CIFAR",
        "preprocessor_name": "clip",
        "data_name": "cifar",
        "model_name": "openai/clip-vit-large-patch14",
        "img_inputs": cifar_img.copy(),
        "txt_inputs": cifar_txt.copy(),
        "target": cifar_labels,
    },
    "clip_b-CIFAR": {
        "exp_name": "clip_b-CIFAR",
        "preprocessor_name": "clip",
        "data_name": "cifar",
        "model_name": "openai/clip-vit-base-patch32",
        "img_inputs": cifar_img.copy(),
        "txt_inputs": cifar_txt.copy(),
        "target": cifar_labels,
    },
    "baseline-CIFAR": {
        "exp_name": "baseline-CIFAR",
        "preprocessor_name": "baseline",
        "data_name": "cifar",
        "img_inputs": cifar_img.copy(),
        "txt_inputs": cifar_txt.copy(),
        "target": cifar_labels,
    },
    "swin_e5-CIFAR": {
        "exp_name": "swin_e5-CIFAR",
        "preprocessor_name": "swin_e5",
        "data_name": "cifar",
        "img_inputs": cifar_img.copy(),
        "txt_inputs": cifar_txt.copy(),
        "target": cifar_labels,
    },
    "clip_l-MSCOCO": {
        "exp_name": "clip_l-MSCOCO",
        "preprocessor_name": "clip",
        "model_name": "openai/clip-vit-large-patch14",
        "data_name": "mscoco",
        "img_inputs": mscoco_img.copy(),
        "txt_inputs": mscoco_txt.copy(),
        "target": None,
    },
    "clip_b-MSCOCO": {
        "exp_name": "clip_b-MSCOCO",
        "preprocessor_name": "clip",
        "model_name": "openai/clip-vit-base-patch32",
        "data_name": "mscoco",
        "img_inputs": mscoco_img.copy(),
        "txt_inputs": mscoco_txt.copy(),
        "target": None,
    },
    "baseline-MSCOCO": {
        "exp_name": "baseline-MSCOCO",
        "preprocessor_name": "baseline",
        "data_name": "mscoco",
        "img_inputs": mscoco_img.copy(),
        "txt_inputs": mscoco_txt.copy(),
        "target": None,
    },
    "swin_e5-MSCOCO": {
        "exp_name": "swin_e5-MSCOCO",
        "preprocessor_name": "swin_e5",
        "data_name": "mscoco",
        "img_inputs": mscoco_img.copy(),
        "txt_inputs": mscoco_txt.copy(),
        "target": None,
    },
    "clip_b-EMNIST": {
        "exp_name": "clip_b-EMNIST",
        "preprocessor_name": "clip",
        "model_name": "openai/clip-vit-base-patch32",
        "data_name": "emnist",
        "img_inputs": emnist_img.copy(),
        "txt_inputs": emnist_txt.copy(),
        "target": emnist_labels,
    },
    "baseline-EMNIST": {
        "exp_name": "baseline-EMNIST",
        "preprocessor_name": "baseline",
        "data_name": "emnist",
        "img_inputs": emnist_img.copy(),
        "txt_inputs": emnist_txt.copy(),
        "target": emnist_labels,
    },
    "swin_e5-EMNIST": {
        "exp_name": "swin_e5-EMNIST",
        "preprocessor_name": "swin_e5",
        "data_name": "emnist",
        "img_inputs": emnist_img.copy(),
        "txt_inputs": emnist_txt.copy(),
        "target": emnist_labels,
    },
    "clip_l-EMNIST": {
        "exp_name": "clip_l-EMNIST",
        "preprocessor_name": "clip",
        "model_name": "openai/clip-vit-large-patch14",
        "data_name": "emnist",
        "img_inputs": emnist_img.copy(),
        "txt_inputs": emnist_txt.copy(),
        "target": emnist_labels,
    },
}

## Scoring

#### Запуск 1

In [12]:
results = None
for exp in exp_setup.keys():
    results = get_clustering_experiment(**exp_setup[exp], df_results=results)
results

exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step


,method-data,silhouette,calinski_harabasz,davies_bouldin_score,rand_score,normalized_mutual_info_score
0,clip_l-CIFAR,0.302555,185.989342,1.623501,0.998585,0.988492
1,clip_b-CIFAR,0.257116,165.227198,1.727127,0.997886,0.983191
2,baseline-CIFAR,0.028960,48.969324,3.406271,0.983230,0.537405
3,swin_e5-CIFAR,0.049104,41.527948,3.405328,0.990817,0.795150
4,clip_l-MSCOCO,0.042772,108.284101,3.711302,0.000000,0.000000
5,clip_b-MSCOCO,0.059150,129.111738,3.377657,0.000000,0.000000
6,baseline-MSCOCO,0.024232,102.453509,3.925560,0.000000,0.000000
7,swin_e5-MSCOCO,-0.019003,47.161403,4.756545,0.000000,0.000000
8,clip_b-EMNIST,0.180860,981.822426,1.945695,0.949457,0.838409
9,baseline-EMNIST,0.090297,653.257323,2.658252,0.879040,0.477956


#### Запуск 2

In [10]:
results = None
for exp in exp_setup.keys():
    results = get_clustering_experiment(**exp_setup[exp], df_results=results)
results

exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step


,method-data,silhouette,calinski_harabasz,davies_bouldin_score,rand_score,normalized_mutual_info_score
0,clip_l-CIFAR,0.317197,192.217042,1.480099,0.999035,0.991337
1,clip_b-CIFAR,0.257235,165.817166,1.723265,0.998324,0.986087
2,baseline-CIFAR,0.028717,48.550119,3.485935,0.982719,0.527603
3,swin_e5-CIFAR,0.047744,40.598232,3.360319,0.990572,0.791719
4,clip_l-MSCOCO,0.045003,108.911483,3.709969,0.000000,0.000000
5,clip_b-MSCOCO,0.061396,124.741750,3.438490,0.000000,0.000000
6,baseline-MSCOCO,0.031797,103.713150,3.947727,0.000000,0.000000
7,swin_e5-MSCOCO,-0.001107,53.997272,4.723514,0.000000,0.000000
8,clip_b-EMNIST,0.166391,951.637574,2.004728,0.963178,0.879171
9,baseline-EMNIST,0.097526,660.999778,2.629184,0.891217,0.505848


#### Запуск 3

In [13]:
results = None
for exp in exp_setup.keys():
    results = get_clustering_experiment(**exp_setup[exp], df_results=results)
results

exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step
exp_name
preprocessor step
preprocessor kmeans step
calculate metrics step


,method-data,silhouette,calinski_harabasz,davies_bouldin_score,rand_score,normalized_mutual_info_score
0,clip_l-CIFAR,0.289647,179.754716,1.720038,0.997886,0.982286
1,clip_b-CIFAR,0.255288,165.176544,1.720279,0.998258,0.984205
2,baseline-CIFAR,0.031676,49.375174,3.418684,0.983480,0.539402
3,swin_e5-CIFAR,0.050967,41.069479,3.403658,0.990199,0.794732
4,clip_l-MSCOCO,0.049044,110.139822,3.886261,0.000000,0.000000
5,clip_b-MSCOCO,0.060398,128.767754,3.506472,0.000000,0.000000
6,baseline-MSCOCO,0.024989,96.900117,4.099135,0.000000,0.000000
7,swin_e5-MSCOCO,-0.014316,51.881336,5.034052,0.000000,0.000000
8,clip_b-EMNIST,0.180113,994.686832,1.917496,0.943845,0.814079
9,baseline-EMNIST,0.090543,658.346805,2.537095,0.878391,0.476906
